# Fixed penalty caps from a revert-rate target

**Inputs** (parameters in the next cell): chain, exclusivity window, target revert rate.
**Output**: two fixed penalty caps in bps of order size — one for **correlated** pairs, one for
**uncorrelated** pairs — plus figures evaluating them over time, per pair, and against the
current fixed native-amount cap.

The model, in three sentences:

1. A solver that won an auction holds exclusivity for `T_EXCL` seconds. If the price moves
   against it by more than the cap during that window, reverting is cheaper than settling — so,
   for one pair, the expected *price-driven* revert rate at a fixed cap `c` is the share of
   historical `T_EXCL`-second price moves worse than `-c`.
2. CoW settlement attempts on the chain give the weights: the expected revert rate of a fixed
   cap is the attempt-weighted average of the per-pair rates.
3. Pairs are **correlated** when both legs sit in the same CoW correlated-token bucket (the CMS
   lists behind the reduced fee), **uncorrelated** otherwise. Each group's cap is the smallest
   `c` whose expected group revert rate is at or below the target.

The notebook is self-contained. It fetches everything it needs and caches it under `../data/`
(delete a cached file to refresh it):

- CoW settlement attempts: `data/{chain}_*.csv`, created via `scripts/fetch_penalties_data.py`
  if missing (needs the analytics-DB credentials from `.env`);
- correlated-token buckets: `https://cms.cow.finance/api/correlated-tokens`;
- token symbols per address: the Coingecko token list for the chain;
- Binance listings (`exchangeInfo`) and 1-second klines for the price history.

Assumptions/simplifications:

1. price-driven reverts only — technical/external reverts sit on top of this target;
2. no bidding response — a revert happens iff the adverse move exceeds the cap;
3. Binance prices (1-second klines) are used as proxy for on-chain executable prices;
4. direct percentile method — the cap is a quantile of historical moves.

In [ ]:
import glob
import json
import os
import subprocess
import sys
import urllib.error
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CHAIN = "base"          # ethereum | gnosis | polygon | bnb | arbitrum | base | avalanche_c
T_EXCL = 8                 # exclusivity window in seconds
TARGET_REVERT_RATE = 0.08   # tolerated price-driven revert rate per settlement attempt

MIN_PAIR_ATTEMPTS = 100     # pairs below this barely move the weighted curve; skip their feeds
COW_START, COW_END = "2026-01-01", "2026-06-22"   # attempt window (only used when fetching)

# days of price history the caps are calibrated on
PRICE_DAYS = [d.strftime("%Y-%m-%d") for d in pd.date_range("2026-05-01", "2026-06-10")]

# chain config (mirrors scripts/fetch_penalties_data.py and the CMS/Coingecko naming)
NATIVE = {"ethereum": "ETH", "arbitrum": "ETH", "base": "ETH", "gnosis": "XDAI",
          "polygon": "POL", "bnb": "BNB", "avalanche_c": "AVAX"}
CMS_NET = {"ethereum": "MAINNET", "arbitrum": "ARBITRUM", "base": "BASE", "gnosis": "GNOSIS",
           "polygon": "POLYGON", "bnb": "BNB", "avalanche_c": "AVALANCHE"}
CG_PLATFORM = {"ethereum": "ethereum", "arbitrum": "arbitrum-one", "base": "base",
               "gnosis": "xdai", "polygon": "polygon-pos", "bnb": "binance-smart-chain",
               "avalanche_c": "avalanche"}

KLINE_CACHE = "../data/binance_klines_1s"
os.makedirs(KLINE_CACHE, exist_ok=True)

## 1. Token pairs traded on CoW

One row of the chain CSV is one settlement attempt (an order inside an auction's winning
solution). We keep the standard frame used across this repo — fill-or-kill, in-market,
penalty-eligible — and count attempts per directed pair `(sell_token, buy_token)`.

Attempts, not USD volume, are the weight: the revert rate we target is a rate *per attempt*, and
attempts are observed for settled and reverted rows alike. `volume_native` and
`penalty_cap_native` are kept per attempt for the comparison with the current cap in section 6.

In [ ]:
files = sorted(glob.glob(f"../data/{CHAIN}_*.csv"))
if not files:   # fetch through the repo's own script (needs DB credentials in ../.env)
    subprocess.run([sys.executable, "../scripts/fetch_penalties_data.py", "--chain", CHAIN,
                    "--start", COW_START, "--end", COW_END], check=True)
    files = sorted(glob.glob(f"../data/{CHAIN}_*.csv"))
COW_FILE = files[-1]

cow = pd.read_csv(COW_FILE, low_memory=False,
                  usecols=["sell_token", "buy_token", "partially_fillable", "is_out_of_market",
                           "is_excluded_from_penalties", "volume_native", "penalty_cap_native"])
cow = cow[cow.partially_fillable.eq(False)
          & cow.is_out_of_market.eq(False)   # rows with NaN flags drop out
          & cow.is_excluded_from_penalties.eq(False)]
cow["sell_token"] = cow.sell_token.str.lower()
cow["buy_token"] = cow.buy_token.str.lower()
att = cow[["sell_token", "buy_token", "volume_native", "penalty_cap_native"]]

pairs = att.groupby(["sell_token", "buy_token"]).size().rename("attempts").reset_index()
print(f"{COW_FILE}: {len(att):,} attempts across {len(pairs):,} directed pairs")

## 2. Classify each pair and map it to a price feed

**Classification** uses CoW's own correlated-token buckets from the CMS (one "Stables" and one
"WETH assets" bucket per chain): a pair is **correlated** iff both legs are in the same bucket.

**Price feed**: every token is resolved to the Binance USDT book that proxies its price, and a
directed pair's price is the ratio `sell_book / buy_book` (the price of USDT is 1). Resolution order per token:

1. its symbol is listed on Binance (`ETH`, `USDC`, `LINK`, ...);
2. its symbol minus a leading `W` is listed (`WETH -> ETH`, `WBTC -> BTC`, `WPOL -> POL`);
3. it is in a CMS bucket → the bucket's **anchor**: the most CoW-traded bucket member that
   itself resolves (so `wstETH -> ETH`, `DAI -> USDC`, gnosis `xDAI -> USDC`, ...).

Symbols come from the CMS buckets themselves, the Coingecko token list for the chain, and a
small native-token config. Two kinds of pairs get no feed and are excluded from the curves but
reported below: pairs where a token resolves to nothing (long-tail), and **same-book pairs**
whose two legs resolve to the same Binance book (`WETH↔wstETH`, `DAI↔USDC`, ...) — the composed
ratio would be constant, which says the pegged cross has no independent Binance price, not that
it cannot move.

Because the pair price is always `sell/buy`, the adverse direction for the solver — the sell
token depreciating against the buy token — is always the **left tail** of the move distribution;
no per-pair orientation is needed.

In [ ]:
def cached_json(path, url):
    if not os.path.exists(path):        # delete the cached file to refresh it
        req = urllib.request.Request(url, headers={"User-Agent": "penalty-research-notebook"})
        with urllib.request.urlopen(req) as r, open(path, "wb") as f:
            f.write(r.read())
    with open(path) as f:
        return json.load(f)


cms = cached_json("../data/cow_correlated_tokens.json",
                  "https://cms.cow.finance/api/correlated-tokens?pagination%5BpageSize%5D=100")
BUCKETS = {b["attributes"]["name"]: {a.lower(): s for a, s in b["attributes"]["tokens"].items()}
           for b in cms["data"] if f"for {CMS_NET[CHAIN]}" in b["attributes"]["name"]}

cg = cached_json(f"../data/tokens_{CG_PLATFORM[CHAIN]}.json",
                 f"https://tokens.coingecko.com/{CG_PLATFORM[CHAIN]}/all.json")
SYMBOL = {t["address"].lower(): t["symbol"] for t in cg["tokens"]}
for tokens in BUCKETS.values():
    SYMBOL.update(tokens)                                   # CMS symbols take precedence
SYMBOL.setdefault("0x" + "e" * 40, NATIVE[CHAIN])           # native-token sentinel

info = cached_json("../data/binance_exchangeinfo.json",
                   "https://api.binance.com/api/v3/exchangeInfo")
BASES = {s["baseAsset"] for s in info["symbols"]
         if s["quoteAsset"] == "USDT" and s["status"] == "TRADING"
         and s["isSpotTradingAllowed"]}


def bucket_of(addr):
    return next((name for name, tokens in BUCKETS.items() if addr in tokens), None)


def binance_asset(sym):
    u = sym.upper()
    if u == "USDT" or u in BASES:       # USDT itself is the quote: constant price 1
        return u
    if len(u) > 2 and u.startswith("W") and u[1:] in BASES:
        return u[1:]                    # wrapped majors: WETH -> ETH, WBTC -> BTC
    return None


# bucket anchor = the most CoW-traded member that resolves to a Binance book
traded = (pd.concat([att.groupby("sell_token").size(), att.groupby("buy_token").size()])
            .groupby(level=0).sum())
ANCHOR = {name: next((binance_asset(tokens[a])
                      for a in sorted(tokens, key=lambda a: -traded.get(a, 0))
                      if binance_asset(tokens[a])), None)
          for name, tokens in BUCKETS.items()}


def resolve(addr):
    """Binance asset whose USDT book proxies this token's price (None if none found)."""
    sym = SYMBOL.get(addr)
    asset = binance_asset(sym) if sym else None
    if asset is None and bucket_of(addr):
        asset = ANCHOR[bucket_of(addr)]
    return asset


label = lambda a: SYMBOL.get(a, a[:8] + "…")
pairs["label"] = [f"{label(s)}→{label(b)}" for s, b in zip(pairs.sell_token, pairs.buy_token)]
pairs["group"] = ["correlated" if bucket_of(s) is not None and bucket_of(s) == bucket_of(b)
                  else "uncorrelated" for s, b in zip(pairs.sell_token, pairs.buy_token)]
asset_s = [resolve(a) for a in pairs.sell_token]
asset_b = [resolve(a) for a in pairs.buy_token]
pairs["asset_s"], pairs["asset_b"] = asset_s, asset_b
pairs["feed"] = [f"{a}/{b}" if a is not None and b is not None and a != b else None
                 for a, b in zip(asset_s, asset_b)]

usable = pairs[pairs.feed.notna() & (pairs.attempts >= MIN_PAIR_ATTEMPTS)].copy()

total = pairs.attempts.sum()
same_book = pairs[pairs.asset_s.notna() & (pairs.asset_s == pairs.asset_b)].attempts.sum()
print(f"buckets: {list(BUCKETS)} — anchors {ANCHOR}")
print(f"calibrating on {usable.attempts.sum() / total:.1%} of attempts "
      f"(usable feed and >= {MIN_PAIR_ATTEMPTS} attempts); "
      f"same-book pairs excluded: {same_book / total:.1%}; "
      f"unresolved long tail: {pairs[pairs.asset_s.isna() | pairs.asset_b.isna()].attempts.sum() / total:.1%}")
print(usable.groupby("group").attempts.sum().to_string())
usable.sort_values("attempts", ascending=False).head(15)[
    ["label", "group", "feed", "attempts"]]

## 3. Price moves over the exclusivity window

For each Binance book: 1-second klines (close per second), forward-filled onto the full per-day
1-second grid. A pair's per-day price grid is the ratio of its two books' grids; the moves are
non-overlapping `T_EXCL`-second windows, `(p_end / p_start - 1) * 1e4` bps.

1s klines were validated against raw tick data for exactly this use in the clustering notebook
(median cap difference < 0.3 bps). The first run downloads roughly 1 GB into
`data/binance_klines_1s/` and caches; later runs are offline. Books with no archive for a day
(e.g. a token listed mid-window) simply skip that day.

In [ ]:
def closes_1s(symbol, day):
    """Close price per second of one UTC day, forward-filled onto the 86400-second grid."""
    path = os.path.join(KLINE_CACHE, f"{symbol}-1s-{day}.zip")
    if os.path.exists(path + ".missing"):
        return None
    if not os.path.exists(path):
        url = f"https://data.binance.vision/data/spot/daily/klines/{symbol}/1s/{symbol}-1s-{day}.zip"
        try:
            urllib.request.urlretrieve(url, path)
        except urllib.error.HTTPError:
            open(path + ".missing", "w").close()
            return None
    with zipfile.ZipFile(path) as z:
        df = pd.read_csv(z.open(z.namelist()[0]), header=None, usecols=[0, 4],
                         names=["ts", "close"])
    if not str(df.ts.iloc[0]).isdigit():                    # newer files ship a header row
        df = df.iloc[1:].astype({"ts": np.int64, "close": float})
    unit = 1_000_000 if df.ts.iloc[0] > 10**14 else 1_000   # timestamps: us (2025+) or ms
    sec_of_day = (df.ts.to_numpy(np.int64) // unit) % 86400
    px = np.full(86400, np.nan)
    px[sec_of_day] = df.close.to_numpy(float)
    return pd.Series(px).ffill().bfill().to_numpy()


def t_moves(grid):
    p = grid[::T_EXCL]
    return (p[1:] / p[:-1] - 1.0) * 1e4     # bps over one exclusivity window


ASSETS = sorted({a for f in usable.feed.unique() for a in f.split("/")})
DAY_MOVES = {f: {} for f in usable.feed.unique()}
for day in PRICE_DAYS:
    grids = {a: np.ones(86400) if a == "USDT" else closes_1s(a + "USDT", day)
             for a in ASSETS}
    for f in DAY_MOVES:
        s, b = f.split("/")
        if grids[s] is not None and grids[b] is not None:
            DAY_MOVES[f][day] = t_moves(grids[s] / grids[b])

MOVES = {f: np.concatenate(list(d.values())) for f, d in DAY_MOVES.items() if d}
dropped = usable[~usable.feed.isin(MOVES)]
if len(dropped):
    print(f"no price data at all, dropped: {sorted(dropped.feed.unique())}")
    usable = usable[usable.feed.isin(MOVES)].copy()
print(f"{len(ASSETS)} Binance books, {len(MOVES)} composed feeds, "
      f"{min(len(m) for m in MOVES.values()):,}+ windows of {T_EXCL}s each")

## 4. Revert rate vs cap, and the two caps

Per pair, the revert rate at cap `c` is the share of its moves below `-c`. The group curve is
the attempt-weighted average of pair curves, and the group's fixed cap is the smallest grid
point where the curve is at or below the target.

(For a single pair this is exactly the target-quantile of its moves with flipped sign — the
direct percentile method of the other cap notebooks. The weighted curve generalizes it to a
group's traffic mix.)

In [ ]:
CAP_GRID = np.arange(0.0, 50.0001, 0.05)    # candidate caps in bps


def revert_curve(moves):
    """Share of moves below -c, for every c in CAP_GRID."""
    return np.searchsorted(np.sort(moves), -CAP_GRID, side="left") / len(moves)


CURVES = {f: revert_curve(m) for f, m in MOVES.items()}

caps, cap_idx = {}, {}
fig, ax = plt.subplots(figsize=(8, 4.5))
for group, rows in usable.groupby("group"):
    weights = rows.attempts / rows.attempts.sum()
    curve = np.sum([w * CURVES[f] for w, f in zip(weights, rows.feed)], axis=0)
    assert curve[-1] <= TARGET_REVERT_RATE, f"{group}: raise the CAP_GRID upper end"
    cap_idx[group] = int(np.argmax(curve <= TARGET_REVERT_RATE))
    caps[group] = CAP_GRID[cap_idx[group]]
    ax.plot(CAP_GRID, curve, label=f"{group}: cap = {caps[group]:.2f} bps")
    ax.axvline(caps[group], color=ax.lines[-1].get_color(), ls=":", lw=1)
ax.axhline(TARGET_REVERT_RATE, color="gray", ls="--", lw=1,
           label=f"target = {TARGET_REVERT_RATE:.0%}")
ax.set(xlabel="fixed penalty cap (bps of order size)",
       ylabel="expected price-driven revert rate",
       title=f"{CHAIN}, T = {T_EXCL}s, attempt-weighted over CoW pairs",
       xlim=(0, 20), ylim=(0, 5 * TARGET_REVERT_RATE))
ax.legend()
plt.show()

In [ ]:
print(f"chain = {CHAIN}, T = {T_EXCL}s, target price-driven revert rate = "
      f"{TARGET_REVERT_RATE:.1%}\n")
for group, cap in sorted(caps.items()):
    print(f"  fixed cap, {group:12s} pairs: {cap:5.2f} bps of order size")

usable["weight"] = usable.attempts / usable.groupby("group").attempts.transform("sum")
usable["rate_at_cap"] = [CURVES[r.feed][cap_idx[r.group]] for r in usable.itertuples()]
usable.sort_values("attempts", ascending=False).head(20)[
    ["label", "group", "feed", "attempts", "weight", "rate_at_cap"]].round(4)

## 5. Evaluating the caps: over time, and per pair

**Over time** — the daily expected revert rate at the (fixed) chosen caps, with pair weights
held constant. Movement in these lines is purely market volatility: the same cap that delivers
the target on average delivers more reverts in turbulent weeks and fewer in calm ones. Large
swings are the argument for an adaptive cap (e.g. "constant fitted k × current volatility estimate",
as in the empirical-approach notebooks) instead of a fixed one.

**Per pair** — the revert rate each pair experiences at its group's cap. Dispersion here is the
cost of one cap per group: quiet majors sit below the target and volatile long-tail pairs far
above it. A wide spread within "uncorrelated" is the case for a third tier (or for per-pair
caps).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for group, rows in usable.groupby("group"):
    num, den = {}, {}
    for r in rows.itertuples():
        for day, mv in DAY_MOVES[r.feed].items():
            num[day] = num.get(day, 0.0) + r.attempts * (mv < -caps[group]).mean()
            den[day] = den.get(day, 0.0) + r.attempts
    daily = pd.Series({pd.Timestamp(d): num[d] / den[d] for d in num}).sort_index()
    ax.plot(daily.index, daily.values, label=f"{group} (cap = {caps[group]:.2f} bps)")
ax.axhline(TARGET_REVERT_RATE, color="gray", ls="--", lw=1,
           label=f"target = {TARGET_REVERT_RATE:.0%}")
ax.set(ylabel="expected price-driven revert rate",
       title=f"{CHAIN}: daily revert rate at the chosen fixed caps")
ax.legend()
fig.autofmt_xdate()
plt.show()

In [ ]:
top = usable.sort_values("attempts", ascending=False).head(25).sort_values("rate_at_cap")
colors = {"correlated": "C0", "uncorrelated": "C1"}

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(range(len(top)), top.rate_at_cap, color=top.group.map(colors))
ax.set_yticks(range(len(top)), top.label)
ax.axvline(TARGET_REVERT_RATE, color="gray", ls="--", lw=1)
ax.set(xlabel="expected price-driven revert rate at the group's cap",
       title=f"{CHAIN}: per-pair revert rate at the fixed caps "
             f"(top 25 pairs by attempts)")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c, label=g) for g, c in colors.items()]
          + [plt.Line2D([], [], color="gray", ls="--",
                        label=f"target = {TARGET_REVERT_RATE:.0%}")])
plt.tight_layout()
plt.show()

## 6. Comparison with the current cap (fixed native amount per chain)

Today the cap is a fixed native-token amount per chain (`penalty_cap_native`, constant in the
extract). In bps of order size it is therefore *size-dependent*: `cap_native / volume_native`.
Small orders get an enormous bps cap, whales a tiny one.

To compare regimes on equal footing we ask, for every mapped attempt: *what price-driven revert
probability does its cap imply?* — reading the attempt's pair curve at the attempt's cap
(current: its individual bps equivalent; proposed: its group's fixed cap). Averaging gives the
expected revert rate under each regime, overall and by order size.

In [ ]:
att2 = att.merge(usable[["sell_token", "buy_token", "group", "feed", "rate_at_cap"]],
                 on=["sell_token", "buy_token"])
att2 = att2[att2.volume_native > 0].copy()
att2["cap_now_bps"] = att2.penalty_cap_native / att2.volume_native * 1e4
att2["p_now"] = np.nan
for f, rows in att2.groupby("feed"):
    att2.loc[rows.index, "p_now"] = np.interp(rows.cap_now_bps, CAP_GRID, CURVES[f])

cap_native = att2.penalty_cap_native.iloc[0] / 1e18
q = att2.cap_now_bps.quantile([0.1, 0.5, 0.9])
print(f"current cap = {cap_native:g} {NATIVE[CHAIN]} per settlement; per attempt that is "
      f"{q[0.5]:.1f} bps of order size (p10 {q[0.1]:.1f}, p90 {q[0.9]:.1f})\n")
print(f"expected price-driven revert rate on the mapped flow ({len(att2):,} attempts):")
print(f"  current fixed native cap   : {att2.p_now.mean():.2%}")
print(f"  proposed two fixed bps caps: {att2.rate_at_cap.mean():.2%}"
      f"   (target {TARGET_REVERT_RATE:.0%})")

att2["decile"] = pd.qcut(att2.volume_native, 10, labels=False, duplicates="drop")
by_size = att2.groupby("decile").agg(size_native=("volume_native", "median"),
                                     current=("p_now", "mean"),
                                     proposed=("rate_at_cap", "mean"))
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(by_size.size_native / 1e18, by_size.current, "o--", color="C0",
        label=f"current cap ({cap_native:g} {NATIVE[CHAIN]}, fixed absolute)")
ax.plot(by_size.size_native / 1e18, by_size.proposed, "o-", color="C1",
        label="proposed caps (fixed bps of size)")
ax.axhline(TARGET_REVERT_RATE, color="gray", ls="--", lw=1,
           label=f"target = {TARGET_REVERT_RATE:.0%}")
ax.set(xscale="log", xlabel=f"order size (decile median, {NATIVE[CHAIN]})",
       ylabel="expected price-driven revert rate",
       title=f"{CHAIN}: revert rate by order size, current vs proposed cap")
ax.legend()
plt.show()

## 7. Caveats — read before quoting the numbers

1. **Price-driven reverts only.** Realized revert rates carry a large non-price component
   (ethereum Jan–Jun 2026: ~11.6% on this same frame), far above any price-driven target. If the
   target is meant for *total* reverts, subtract the external share `p_ext` first: price
   allowance = `(q - p_ext) / (1 - p_ext)` (8pct notebook, section 7).
2. **Same-book pairs are excluded from the curves** (`WETH↔wstETH`, `DAI↔USDC`, ...): their legs
   resolve to one Binance book, so the model has no independent price for the cross. The listed
   pegged books that do exist (`USDCUSDT`, `WBETHETH`) are tick-bound at ~0.1 bps, so their
   price-driven revert risk is a tick floor — while realized WETH↔wstETH revert rates are ~50%,
   i.e. overwhelmingly non-price-driven. A small correlated cap is what the price model
   justifies; it will not deter whatever actually drives those reverts.
3. **CMS buckets are taken as-is.** The stables bucket mixes USD stables, EUR stables and
   tokenized equities, so e.g. a EUR↔USD stable pair counts as correlated while carrying real FX
   volatility; there is no BTC bucket, so `WBTC↔cbBTC` counts as uncorrelated (and is excluded
   as a same-book pair anyway).
4. **Synthetic crosses overstate volatility** (independent leg noise adds instead of cancelling
   — `synth_vs_direct_vol` notebook), so volatile/volatile pairs err toward a conservative cap.
5. **No bidding response.** Solvers shade bids when caps change (tie-equilibrium model in the
   `revert-rate-vs-cap` branch and 8pct section 7); over 1–10 bps caps the bid premium stays
   below ~0.5 bps, so the effect is second-order and omitted.
6. **Coverage.** Only pairs with a resolvable feed and enough attempts enter; the printed
   coverage line says how much flow that is. The unresolved long tail is likely *more* volatile
   than the mapped flow — see the per-pair figure for how wide the dispersion already is inside
   the mapped set (the case for a third tier, and the clustering notebook's territory).
7. **One sample period, one flow mix.** Prices from a fixed window (single volatility regime;
   see the over-time figure for how much the rate breathes within it); weights from realized
   attempts over the extract window. Walk-forward stability of percentile caps is tested in the
   empirical-approach notebooks.
8. **Current-cap comparison** covers the mapped flow only; revert probabilities are read off
   curves that end at 50 bps (attempts whose current cap exceeds that — small orders — are
   assigned the 50 bps rate, understating nothing material); attempts with missing
   `volume_native` are dropped.

Relation to the other notebooks: this one produces *fixed* per-group caps;
`penalty_cap_empirical_approach{,_8pct}` produce per-pair adaptive caps (`k·σ·√T` and rolling
percentiles); `solver_revert_option_pricing` derives the theory linking cap and revert
probability; `clustering_analysis_T26` assigns caps across the whole token universe;
`synth_vs_direct_vol` quantifies the synthetic-feed error used in caveat 4.